In [44]:
# importing the required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import kmapper as km
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from umap import UMAP
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import os
import joblib

In [45]:
# importing preprocessed data
combined_df = pd.read_csv("../data/combined_cleaned_encoded.csv")

In [46]:
# Scaling 
# Select numerical columns for scaling
num_cols = ['age', 'tenure', 'monthlycharges']

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on numeric columns and transform them
df_scaled = combined_df.copy()
df_scaled[num_cols] = scaler.fit_transform(combined_df[num_cols])

print(df_scaled.head())

      customer_id  gender       age    tenure  monthlycharges  paymentmethod  \
0  TEL_7590-VHVEG       0 -0.371853 -1.511772       -1.155154              5   
1  TEL_5575-GNVDE       1 -0.103853  0.076123       -0.136535              6   
2  TEL_3668-QPYBK       1 -1.376852 -1.463654       -0.253056              6   
3  TEL_7795-CFOCW       1 -1.443852  0.605421       -0.687190              0   
4  TEL_9237-HQITU       0 -1.577852 -1.463654        0.380292              5   

   churn  industry  
0    0.0         2  
1    0.0         2  
2    1.0         2  
3    0.0         2  
4    1.0         2  


In [9]:
#spliting into features (X) and target (Y)
y = combined_df['churn'] #column named 5 is churn column
X = combined_df.drop(columns=['churn']) 

In [10]:
#train-test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=Y)

## TDA

In [17]:
#dropping the customer_id column since it is not a numerical column
X_train = X_train.drop(columns=["customer_id"])

In [33]:
#dropping the customer_id column since it is not a numerical column
X_test = X_test.drop(columns=["customer_id"])

In [18]:
mapper = km.KeplerMapper(verbose=1)

lens_train = mapper.fit_transform(
    X_train,
    projection=UMAP(n_components=2, random_state=42)
)

graph_train = mapper.map(
    lens_train,
    X_train,
    clusterer=KMeans(n_clusters=4),
    cover=km.Cover(n_cubes=10, perc_overlap=0.3)
)


KeplerMapper(verbose=1)
..Composing projection pipeline of length 1:
	Projections: UMAP(random_state=42)
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (16903, 6)

..Projecting data using: 
	UMAP(random_state=42, verbose=1)

UMAP(n_jobs=1, random_state=42, verbose=1)
Mon Nov 17 14:09:29 2025 Construct fuzzy simplicial set
Mon Nov 17 14:09:29 2025 Finding Nearest Neighbors
Mon Nov 17 14:09:29 2025 Building RP forest with 12 trees


C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Mon Nov 17 14:09:34 2025 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	Stopping threshold met -- exiting after 2 iterations
Mon Nov 17 14:09:42 2025 Finished Nearest Neighbor Search
Mon Nov 17 14:09:45 2025 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Mon Nov 17 14:09:58 2025 Finished embedding

..Scaling with: MinMaxScaler()

Mapping on data shaped (16903, 6) using lens shaped (16903, 2)

Creating 100 hypercubes.

Created 1396 edges and 364 nodes in 0:00:04.000560.


In [19]:
#convert TDA graph into TRAIN features

train_nodes = graph_train['nodes']
tda_train = np.zeros((X_train.shape[0], len(train_nodes)))

for i, node_samples in enumerate(train_nodes.values()):
    tda_train[node_samples, i] = 1


In [28]:
lens_test = mapper.transform(X_test)


AttributeError: 'KeplerMapper' object has no attribute 'transform'

In [25]:
#Generate TDA features for TEST using SAME mapper

#project test using trained UMAP
reducer = UMAP(n_components=2, random_state=42)
lens_train = reducer.fit_transform(X_train)
#lens_test = reducer.transform(X_test)

#Compute node centers
#node_centers = []

#for i, node_samples in enumerate(nodes.values()):
 #   center = lens_train[node_samples].mean(axis=0)
  #  node_centers.append(center)

#node_centers = np.array(node_centers)

#assign each test point to the closest node
#closest_nodes, _ = pairwise_distances_argmin_min(lens_test, node_centers)

#tda_test = np.zeros((X_test.shape[0], len(nodes)))
#tda_test[np.arange(X_test.shape[0]), closest_nodes] = 1


C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [29]:
import umap

umap_model = umap.UMAP(n_components=2, random_state=42)
lens_train = umap_model.fit_transform(X_train)


C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [30]:
mapper = km.KeplerMapper(verbose=1)

graph_train = mapper.map(
    lens_train,
    X_train,
    clusterer=KMeans(n_clusters=4),
    cover=km.Cover(n_cubes=10, perc_overlap=0.3)
)


KeplerMapper(verbose=1)
Mapping on data shaped (16903, 6) using lens shaped (16903, 2)

Creating 100 hypercubes.

Created 1420 edges and 364 nodes in 0:00:00.573860.


In [31]:
train_nodes = graph_train['nodes']
tda_train = np.zeros((X_train.shape[0], len(train_nodes)))

for i, node_samples in enumerate(train_nodes.values()):
    tda_train[node_samples, i] = 1


In [34]:
lens_test = umap_model.transform(X_test)


In [35]:
# Create an empty matrix for test TDA features
tda_test = np.zeros((X_test.shape[0], len(train_nodes)))

# Compute node centers
node_centers = []
for node_samples in train_nodes.values():
    node_centers.append(np.mean(lens_train[node_samples], axis=0))
node_centers = np.array(node_centers)

# Distance between test points and nodes
from sklearn.metrics.pairwise import euclidean_distances
dist_matrix = euclidean_distances(lens_test, node_centers)

# Assign nearest node
nearest_node = np.argmin(dist_matrix, axis=1)
for i, node in enumerate(nearest_node):
    tda_test[i, node] = 1


In [36]:
X_train_final = np.hstack([X_train, tda_train])
X_test_final = np.hstack([X_test, tda_test])


In [41]:
#Logistic Regression

log_reg = LogisticRegression(max_iter=1000, random_state=42) #creating the model
log_reg.fit(X_train_final, y_train) #training the model (.fit() means learn from the dataset)
y_pred_lr = log_reg.predict(X_test_final) #make predictions
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr)) #evaluate the model's performance
print(classification_report(y_test, y_pred_lr)) #metrics report

Logistic Regression Accuracy: 0.6102697586370089
              precision    recall  f1-score   support

         0.0       0.64      0.78      0.71      2514
         1.0       0.53      0.36      0.43      1712

    accuracy                           0.61      4226
   macro avg       0.58      0.57      0.57      4226
weighted avg       0.60      0.61      0.59      4226



C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [43]:
#Random Forest

rf = RandomForestClassifier(n_estimators=200, random_state=42) #creating the model
rf.fit(X_train_final, y_train) #training the model (.fit() means learn from the dataset)
y_pred_rf = rf.predict(X_test_final) #make predictions
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf)) #evaluate the model's performance
print(classification_report(y_test, y_pred_rf)) #metrics report

Random Forest Accuracy: 0.6710837671557028
              precision    recall  f1-score   support

         0.0       0.70      0.77      0.74      2514
         1.0       0.61      0.52      0.56      1712

    accuracy                           0.67      4226
   macro avg       0.66      0.65      0.65      4226
weighted avg       0.67      0.67      0.67      4226



In [37]:
#XGBOOST
from xgboost import XGBClassifier

model = XGBClassifier()
model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)


In [38]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[1941  573]
 [ 788  924]]
              precision    recall  f1-score   support

         0.0       0.71      0.77      0.74      2514
         1.0       0.62      0.54      0.58      1712

    accuracy                           0.68      4226
   macro avg       0.66      0.66      0.66      4226
weighted avg       0.67      0.68      0.67      4226



## Old tda

In [47]:
#dropping the customer_id column since it is not a numerical column
df_scaled = df_scaled.drop(columns=["customer_id"])

In [48]:
#preparing scaled data (converting from pandas to numpy to get numerical matrix form)
df_data = df_scaled.values

In [49]:
#creating mapper object (verbose=1 shows progess messages)
mapper = km.KeplerMapper(verbose=1)

KeplerMapper(verbose=1)


In [50]:
#choosing the projection
#we take the scaled dataset, reduce it to 2D using TSNE and it gives projected points so Mapper can 
#build a topological graph
projected = mapper.fit_transform(df_data, projection=TSNE(n_components=2))

..Composing projection pipeline of length 1:
	Projections: TSNE()
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (21129, 7)

..Projecting data using: 
	TSNE(verbose=1)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 21129 samples in 0.043s...
[t-SNE] Computed neighbors for 21129 samples in 1.994s...
[t-SNE] Computed conditional probabilities for sample 1000 / 21129
[t-SNE] Computed conditional probabilities for sample 2000 / 21129
[t-SNE] Computed conditional probabilities for sample 3000 / 21129
[t-SNE] Computed conditional probabilities for sample 4000 / 21129
[t-SNE] Computed conditional probabilities for sample 5000 / 21129
[t-SNE] Computed conditional probabilities for sample 6000 / 21129
[t-SNE] Computed conditional probabilities for sample 7000 / 21129
[t-SNE] Computed conditional probabilities for sample 8000 / 21129
[t-SNE] Computed conditional probabilities for sample 9000 / 21129
[t-SNE] Computed conditional probabilities for sample

In [51]:
#creating the topological network
graph=mapper.map(projected, df_data, clusterer=KMeans(n_clusters=4), cover=km.Cover(n_cubes=10, perc_overlap=0.3))

Mapping on data shaped (21129, 7) using lens shaped (21129, 2)

Creating 100 hypercubes.

Created 1181 edges and 340 nodes in 0:00:01.348021.


In [63]:
#extracting TDA features

#graph['nodes'] includes list of sample indices in each node
nodes = graph['nodes']

#creating a feature matrix where each sample gets a binary vector for node members
tda_features = np.zeros((df_data.shape[0], len(nodes)))

for i, node_samples in enumerate(nodes.values()):
    tda_features[node_samples, i] = 1 #marking 1 if the sample belongs to this node


#turning the above into one-hot encoding
tda_features_df = pd.DataFrame(tda_features, columns=[f"tda_node_{i}" for i in range(len(nodes))])

#combining with original data
df_final = pd.concat([pd.DataFrame(df_data).reset_index(drop=True), tda_features_df.reset_index(drop=True)], axis=1)

In [64]:
df_final.columns = df_final.columns.astype(str)


In [65]:
print(df_final.columns)

Index(['0', '1', '2', '3', '4', '5', '6', 'tda_node_0', 'tda_node_1',
       'tda_node_2',
       ...
       'tda_node_330', 'tda_node_331', 'tda_node_332', 'tda_node_333',
       'tda_node_334', 'tda_node_335', 'tda_node_336', 'tda_node_337',
       'tda_node_338', 'tda_node_339'],
      dtype='object', length=347)


In [66]:
#spliting into features (X) and target (Y)
Y = df_final["5"] #column named 5 is churn column
X = df_final.drop(columns=["5"])

In [67]:
#train-test split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [68]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import os
import joblib

In [69]:
#Logistic Regression

log_reg = LogisticRegression(max_iter=1000, random_state=42) #creating the model
log_reg.fit(X_train, Y_train) #training the model (.fit() means learn from the dataset)
y_pred_lr = log_reg.predict(X_test) #make predictions
print("Logistic Regression Accuracy:", accuracy_score(Y_test, y_pred_lr)) #evaluate the model's performance
print(classification_report(Y_test, y_pred_lr)) #metrics report

Logistic Regression Accuracy: 0.9673450070989115
              precision    recall  f1-score   support

         0.0       0.97      0.98      0.97      2514
         1.0       0.97      0.95      0.96      1712

    accuracy                           0.97      4226
   macro avg       0.97      0.96      0.97      4226
weighted avg       0.97      0.97      0.97      4226



In [70]:
#Random Forest

rf = RandomForestClassifier(n_estimators=200, random_state=42) #creating the model
rf.fit(X_train, Y_train) #training the model (.fit() means learn from the dataset)
y_pred_rf = rf.predict(X_test) #make predictions
print("Random Forest Accuracy:", accuracy_score(Y_test, y_pred_rf)) #evaluate the model's performance
print(classification_report(Y_test, y_pred_rf)) #metrics report

Random Forest Accuracy: 0.9758637008991955
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      2514
         1.0       0.98      0.96      0.97      1712

    accuracy                           0.98      4226
   macro avg       0.98      0.97      0.97      4226
weighted avg       0.98      0.98      0.98      4226



In [71]:
# XGBoost

xgb = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=42) #creating the model
xgb.fit(X_train, Y_train) #training the model (.fit() means learn from the dataset)
y_pred_xgb = xgb.predict(X_test) #make predictions
print("XGBoost Accuracy:", accuracy_score(Y_test, y_pred_xgb)) #evaluate the model's performance
print(classification_report(Y_test, y_pred_xgb)) #metrics report

C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:18:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.9716043539990534
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      2514
         1.0       0.98      0.95      0.96      1712

    accuracy                           0.97      4226
   macro avg       0.97      0.97      0.97      4226
weighted avg       0.97      0.97      0.97      4226



In [72]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Assume these are your trained models
# logistic_regression, random_forest, xgboost_model
# Make sure they are fitted already

# Create a voting ensemble (soft voting uses predicted probabilities)
ensemble_model = VotingClassifier(
    estimators=[
        ('lr', log_reg),
        ('rf', rf),
        ('xgb', xgb)
    ],
    voting='soft'  # use 'hard' for majority vote
)

# Fit ensemble on training data
ensemble_model.fit(X_train, Y_train)

# Predict on test data
y_pred_ensemble = ensemble_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(Y_test, y_pred_ensemble)
cm = confusion_matrix(Y_test, y_pred_ensemble)
report = classification_report(Y_test, y_pred_ensemble)

print(f"Ensemble Accuracy: {accuracy:.4f}")
print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(report)


C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:18:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Ensemble Accuracy: 0.9742
Confusion Matrix:
[[2485   29]
 [  80 1632]]
Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      2514
         1.0       0.98      0.95      0.97      1712

    accuracy                           0.97      4226
   macro avg       0.98      0.97      0.97      4226
weighted avg       0.97      0.97      0.97      4226



In [75]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -----------------------------------------
# Base models (these should already be trained)
# -----------------------------------------
# rf  = RandomForestClassifier(...).fit(X_train, y_train)
# lr  = LogisticRegression(...).fit(X_train, y_train)
# xgb = XGBClassifier(...).fit(X_train, y_train)

# IMPORTANT: If not trained already, remove .fit() above and let stacking train them.

# -----------------------------------------
# Stacking Ensemble
# -----------------------------------------
estimators = [
    ('lr', log_reg),
    ('rf', rf),
    ('xgb', xgb)
]

# Meta-model (best choice for classification stacking)
meta_model = LogisticRegression(max_iter=200, class_weight='balanced')

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    stack_method='predict_proba',
    passthrough=True   # gives meta-model BOTH original features + base model outputs
)

# Train stacking model
stack_model.fit(X_train, Y_train)

# Predict
y_pred_stack = stack_model.predict(X_test)

# Evaluate
acc = accuracy_score(Y_test, y_pred_stack)
cm  = confusion_matrix(Y_test, y_pred_stack)
rep = classification_report(Y_test, y_pred_stack)

print("Stacking Ensemble Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", rep)


C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:22:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:23:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:23:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\KARISHMA\Desktop\customer-churn-prediction\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:23:28] WARNING: 

Stacking Ensemble Accuracy: 0.9756270705158543

Confusion Matrix:
 [[2472   42]
 [  61 1651]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.98      0.98      0.98      2514
         1.0       0.98      0.96      0.97      1712

    accuracy                           0.98      4226
   macro avg       0.98      0.97      0.97      4226
weighted avg       0.98      0.98      0.98      4226



In [29]:
#downloading the final dataset into data folder

save_path = "../data"

os.makedirs(save_path, exist_ok=True)

df_final.to_csv(os.path.join(save_path, "combined_cleaned_TDA.csv"), index=False)

In [31]:
#saving scaler
joblib.dump(scaler, "scaler.pkl")
print("Scaler saved!")

Scaler saved!


In [34]:
# Save the lens
#joblib.dump(lens, "models/lens.pkl")

# Save the cover
#joblib.dump(cover, "cover.pkl")

# Save the graph
joblib.dump(graph, "graph.pkl")

print("KMapper components saved!")


KMapper components saved!
